# 0. Install and Import Dependencies

In [ ]:
!pip install mediapipe opencv-python

In [ ]:
import mediapipe as mp
import cv2

In [ ]:
mp_drawing = mp.solutions.drawing_utils
mp_face_mesh = mp.solutions.face_mesh
mp_holistic = mp.solutions.holistic

# 1. Get Realtime Webcam Feed

In [ ]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    cv2.imshow('Raw Webcam Feed', frame)
    
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
cap.release()
cv2.destroyAllWindows()

# 2. Make Detections from Feed
1. Detect Facial Landmarks
2. Detect Hand Poses
3. Detect Body Poses

In [ ]:
import cv2
import mediapipe as mp

# Initialize Mediapipe components
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic
mp_face_mesh = mp.solutions.face_mesh

In [ ]:
# import cv2
# import mediapipe as mp
#
# # Initialize Mediapipe components
# mp_drawing = mp.solutions.drawing_utils
# mp_holistic = mp.solutions.holistic
# mp_face_mesh = mp.solutions.face_mesh

# Access the webcam
cap = cv2.VideoCapture(0)

# Initialize Holistic and FaceMesh models
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic, \
     mp_face_mesh.FaceMesh(max_num_faces=2, min_detection_confidence=0.5, min_tracking_confidence=0.5) as face_mesh:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert the frame color from BGR to RGB for MediaPipe processing
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Process the image for holistic detection
        holistic_results = holistic.process(image)

        # Process the image for face mesh detection
        face_mesh_results = face_mesh.process(image)

        # Convert the image color back to BGR for OpenCV rendering
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Draw detailed face mesh landmarks (FaceMesh)
        if face_mesh_results.multi_face_landmarks:
            for face_landmarks in face_mesh_results.multi_face_landmarks:
                mp_drawing.draw_landmarks(
                    image, face_landmarks, mp_face_mesh.FACEMESH_TESSELATION,
                    mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=1, circle_radius=1),
                    mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=1, circle_radius=1)
                )

        # Draw holistic face landmarks (Holistic - only points, no connections)
        if holistic_results.face_landmarks:
            mp_drawing.draw_landmarks(
                image, holistic_results.face_landmarks, None,  # No connections for Holistic
                mp_drawing.DrawingSpec(color=(80, 110, 10), thickness=1, circle_radius=1),
                mp_drawing.DrawingSpec(color=(80, 256, 121), thickness=1, circle_radius=1)
            )

        # Draw holistic hand landmarks (Holistic - hand keypoints)
        if holistic_results.left_hand_landmarks:
            mp_drawing.draw_landmarks(
                image, holistic_results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2, circle_radius=2)
            )

        if holistic_results.right_hand_landmarks:
            mp_drawing.draw_landmarks(
                image, holistic_results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2, circle_radius=2)
            )

        # Draw holistic pose landmarks (Holistic - body pose)
        if holistic_results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image, holistic_results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2)
            )

        # Display the image
        cv2.imshow('Holistic + Face Mesh', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


In [ ]:
cap.release()
cv2.destroyAllWindows()

In [1]:
import cv2
import mediapipe as mp
import face_recognition

# Initialize Mediapipe components
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic
mp_face_mesh = mp.solutions.face_mesh

# Access the webcam
cap = cv2.VideoCapture(0)

# Initialize Holistic and FaceMesh models
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic, \
     mp_face_mesh.FaceMesh(max_num_faces=2, min_detection_confidence=0.5, min_tracking_confidence=0.5) as face_mesh:

    # Known face encodings and names (you can add more here)
    known_face_encodings = []
    known_face_names = []

    # Example: Load a known face (you can add your own faces here)
    person_image = face_recognition.load_image_file("face/wangyongji.jpg")
    person_encoding = face_recognition.face_encodings(person_image)[0]
    known_face_encodings.append(person_encoding)
    known_face_names.append("ww")  # Assign a name to this face

    # Flag to track if a face has been recognized
    face_recognized = False
    recognized_name = "Unknown"  # Default name if no match is found

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert the frame color from BGR to RGB for MediaPipe processing
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Process the image for holistic detection
        holistic_results = holistic.process(image)

        # Process the image for face mesh detection
        face_mesh_results = face_mesh.process(image)

        # Convert the image color back to BGR for OpenCV rendering
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Detect faces in the frame using face_recognition
        face_locations = face_recognition.face_locations(image)  # Detect face locations
        face_encodings = face_recognition.face_encodings(image, face_locations)  # Get encodings of faces

        if face_locations:
            if not face_recognized:  # Only recognize the face once
                # Compare the detected faces with known faces
                for face_encoding in face_encodings:
                    matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
                    name = "Unknown"  # Default name if no match is found

                    if True in matches:
                        first_match_index = matches.index(True)
                        name = known_face_names[first_match_index]

                    recognized_name = name  # Update the recognized name
                    face_recognized = True  # Set the flag to True after recognition

        # If no faces are detected, reset the face_recognized flag
        else:
            face_recognized = False
            recognized_name = "Unknown"

        # Draw the recognized face name
        for (top, right, bottom, left) in face_locations:
            cv2.rectangle(image, (left, top), (right, bottom), (0, 255, 0), 2)  # Green box
            cv2.putText(image, recognized_name, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

        # Draw detailed face mesh landmarks (FaceMesh)
        if face_mesh_results.multi_face_landmarks:
            for face_landmarks in face_mesh_results.multi_face_landmarks:
                mp_drawing.draw_landmarks(
                    image, face_landmarks, mp_face_mesh.FACEMESH_TESSELATION,
                    mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=1, circle_radius=1),
                    mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=1, circle_radius=1)
                )

        # Draw holistic face landmarks (Holistic - only points, no connections)
        if holistic_results.face_landmarks:
            mp_drawing.draw_landmarks(
                image, holistic_results.face_landmarks, None,  # No connections for Holistic
                mp_drawing.DrawingSpec(color=(80, 110, 10), thickness=1, circle_radius=1),
                mp_drawing.DrawingSpec(color=(80, 256, 121), thickness=1, circle_radius=1)
            )

        # Draw holistic hand landmarks (Holistic - hand keypoints)
        if holistic_results.left_hand_landmarks:
            mp_drawing.draw_landmarks(
                image, holistic_results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2, circle_radius=2)
            )

        if holistic_results.right_hand_landmarks:
            mp_drawing.draw_landmarks(
                image, holistic_results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2, circle_radius=2)
            )

        # Draw holistic pose landmarks (Holistic - body pose)
        if holistic_results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image, holistic_results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2)
            )

        # Display the image
        cv2.imshow('Holistic + Face Mesh + Face Recognition', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

In [2]:
cap.release()
cv2.destroyAllWindows()